In [1]:
import requests
import pandas as pd

# Test 1 — baseline: what do we get with no filters?
resp = requests.get("https://data-api.polymarket.com/trades", params={"limit": 5})
data = resp.json()
df = pd.DataFrame(data)
print("Columns:", list(df.columns))
print("\nTimestamp range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())
print("\nSample titles:")
print(df["title"].tolist())

Columns: ['proxyWallet', 'side', 'asset', 'conditionId', 'size', 'price', 'timestamp', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'outcomeIndex', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'transactionHash']

Timestamp range:
1780350227 to 1780350227

Sample titles:
['Ethereum Up or Down - June 1, 5:40PM-5:45PM ET', 'Dogecoin Up or Down - June 1, 5:40PM-5:45PM ET', 'XRP Up or Down - June 1, 5:30PM-5:45PM ET', 'Solana Up or Down - June 1, 5:40PM-5:45PM ET', 'Ethereum Up or Down - June 1, 5:40PM-5:45PM ET']


In [2]:
# Test 2 — does offset give us older trades?
resp_offset = requests.get(
    "https://data-api.polymarket.com/trades",
    params={"limit": 5, "offset": 1000}
)
data_offset = resp_offset.json()
df_offset = pd.DataFrame(data_offset)
import datetime
print("With offset=1000:")
print("Timestamps:", df_offset["timestamp"].tolist())
print("Titles:", df_offset["title"].tolist())

# Test 3 — does a before/after timestamp filter work?
resp_ts = requests.get(
    "https://data-api.polymarket.com/trades",
    params={"limit": 5, "before": 1780000000}  # a few days ago
)
print("\nWith before= timestamp filter:")
print("Status:", resp_ts.status_code)
print(resp_ts.json())

With offset=1000:
Timestamps: [1780350525, 1780350525, 1780350525, 1780350525, 1780350525]
Titles: ['Bitcoin Up or Down - June 1, 5:45PM-5:50PM ET', 'Bitcoin Up or Down - June 1, 5:45PM-5:50PM ET', 'Bitcoin Up or Down - June 1, 5:45PM-6:00PM ET', 'Bitcoin Up or Down - June 1, 5:45PM-5:50PM ET', 'Bitcoin Up or Down - June 1, 5:45PM-5:50PM ET']

With before= timestamp filter:
Status: 200
[{'proxyWallet': '0x400c31b5f68aa6abc27395224be5c49264a0bc73', 'side': 'BUY', 'asset': '61457157235797509785653353305656428496297766909975388180982130331560747584823', 'conditionId': '0xadf8baa20c00415fd2ae799edfd8b6197ab20a3af6ccf8a9827304561b77327a', 'size': 204.0612, 'price': 0.98, 'timestamp': 1780350532, 'title': 'Bitcoin Up or Down - June 1, 5:45PM-5:50PM ET', 'slug': 'btc-updown-5m-1780350300', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'eventSlug': 'btc-updown-5m-1780350300', 'outcome': 'Up', 'outcomeIndex': 0, 'name': '0x400C31b5F68Aa6aBc27395224Be5C49264A0B

In [4]:
# Test 4 — Polymarket Gamma API timeseries endpoint
# Try fetching historical price data for a known market

# Use one of the conditionIds we've seen
test_condition_id = "0x29283b56d3eec6d1d89fff51793d9d2e01579d2379b1d3e9ebda175d3561e809"  # Phil Murphy 2028

# Try the timeseries endpoint
resp = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": test_condition_id,
        "interval": "1d",
        "fidelity": 60
    }
)
print("prices-history status:", resp.status_code)
print(resp.text[:500])

prices-history status: 200
{"history":[]}



In [5]:
# Test 5 — try Gamma API timeseries and CLOB timeseries variants
endpoints = [
    ("https://clob.polymarket.com/prices-history", {"market": test_condition_id, "interval": "1h", "fidelity": 60}),
    ("https://clob.polymarket.com/prices-history", {"market": test_condition_id, "interval": "all", "fidelity": 60}),
    ("https://gamma-api.polymarket.com/markets", {"id": test_condition_id}),
    ("https://clob.polymarket.com/timeseries", {"market": test_condition_id, "interval": "1d"}),
]

for url, params in endpoints:
    resp = requests.get(url, params=params)
    print(f"\n{url}")
    print(f"Params: {params}")
    print(f"Status: {resp.status_code}")
    print(resp.text[:300])


https://clob.polymarket.com/prices-history
Params: {'market': '0x29283b56d3eec6d1d89fff51793d9d2e01579d2379b1d3e9ebda175d3561e809', 'interval': '1h', 'fidelity': 60}
Status: 200
{"history":[]}


https://clob.polymarket.com/prices-history
Params: {'market': '0x29283b56d3eec6d1d89fff51793d9d2e01579d2379b1d3e9ebda175d3561e809', 'interval': 'all', 'fidelity': 60}
Status: 200
{"history":[]}


https://gamma-api.polymarket.com/markets
Params: {'id': '0x29283b56d3eec6d1d89fff51793d9d2e01579d2379b1d3e9ebda175d3561e809'}
Status: 422
{"type":"validation error","error":"\"id\" has a wrong value"}


https://clob.polymarket.com/timeseries
Params: {'market': '0x29283b56d3eec6d1d89fff51793d9d2e01579d2379b1d3e9ebda175d3561e809', 'interval': '1d'}
Status: 404
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
<script>(function(){function c(){var b=a.contentDocument||a.contentWindow.document;if(b){var d=b.createElement('script');d.i

In [6]:
# Test 6 — try asset ID instead of conditionId, and find a resolved market
# The asset field in trades may be the correct identifier for price history

test_asset = "114507466536734360808195115190160357255467689128176113085692265197019898971527"  # Phil Murphy No token

resp1 = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={"market": test_asset, "interval": "all", "fidelity": 60}
)
print("Using asset ID:")
print(f"Status: {resp1.status_code}")
print(resp1.text[:300])

# Also find a resolved market from Gamma API
resp2 = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={"closed": "true", "limit": 3}
)
data2 = resp2.json()
print("\nResolved market sample:")
for m in data2[:2]:
    print(f"  conditionId: {m.get('conditionId')}")
    print(f"  question: {m.get('question')}")
    print(f"  clobTokenIds: {m.get('clobTokenIds')}")
    print()

Using asset ID:
Status: 200
{"history":[{"t":1777672806,"p":0.9935},{"t":1777676407,"p":0.9935},{"t":1777680013,"p":0.9935},{"t":1777683606,"p":0.9935},{"t":1777687210,"p":0.9935},{"t":1777690811,"p":0.9935},{"t":1777694407,"p":0.9935},{"t":1777698009,"p":0.9935},{"t":1777701610,"p":0.9935},{"t":1777705225,"p":0.9935},{"t":177

Resolved market sample:
  conditionId: 0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa00481764d320e7415f7a9
  question: Will Joe Biden get Coronavirus before the election?
  clobTokenIds: ["53135072462907880191400140706440867753044989936304433583131786753949599718775", "60869871469376321574904667328762911501870754872924453995477779862968218702336"]

  conditionId: 0x44f10d1cd5aaed4b7ae0b5edb76790f54f45dc0bcaa86831c83d865c774fbb90
  question: Will Airbnb begin publicly trading before Jan 1, 2021?
  clobTokenIds: ["23957885615115430922384185661294483989521212430808224513177413172438775950057", "440659171691388154510320589265569600333745571378792500750915453224369318

In [7]:
# Test 7 — get full price history for a resolved market token
token_id = "53135072462907880191400140706440867753044989936304433583131786753949599718775"

resp = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={"market": token_id, "interval": "all", "fidelity": 60}
)
data = resp.json()
history = data.get("history", [])
print(f"Data points: {len(history)}")

if history:
    df = pd.DataFrame(history)
    df["t"] = pd.to_datetime(df["t"], unit="s", utc=True)
    print(f"Date range: {df['t'].min()} to {df['t'].max()}")
    print(f"Price range: {df['p'].min()} to {df['p'].max()}")
    print(df.head(10))

Data points: 0


In [8]:
# Test 8 — check how much history we actually have for the Phil Murphy token
test_asset = "114507466536734360808195115190160357255467689128176113085692265197019898971527"

resp = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={"market": test_asset, "interval": "all", "fidelity": 60}
)
data = resp.json()
history = data.get("history", [])
print(f"Data points: {len(history)}")

if history:
    df = pd.DataFrame(history)
    df["t"] = pd.to_datetime(df["t"], unit="s", utc=True)
    print(f"Date range: {df['t'].min()} to {df['t'].max()}")
    print(f"Price range: {df['p'].min()} to {df['p'].max()}")
    print(f"\nFirst 5 rows:")
    print(df.head())
    print(f"\nLast 5 rows:")
    print(df.tail())

Data points: 743
Date range: 2026-05-01 22:00:06+00:00 to 2026-06-01 21:55:19+00:00
Price range: 0.9935 to 0.9935

First 5 rows:
                          t       p
0 2026-05-01 22:00:06+00:00  0.9935
1 2026-05-01 23:00:07+00:00  0.9935
2 2026-05-02 00:00:13+00:00  0.9935
3 2026-05-02 01:00:06+00:00  0.9935
4 2026-05-02 02:00:10+00:00  0.9935

Last 5 rows:
                            t       p
738 2026-06-01 18:00:08+00:00  0.9935
739 2026-06-01 19:00:06+00:00  0.9935
740 2026-06-01 20:00:07+00:00  0.9935
741 2026-06-01 21:00:08+00:00  0.9935
742 2026-06-01 21:55:19+00:00  0.9935


In [9]:
# Test 9 — discover available tags
resp = requests.get("https://gamma-api.polymarket.com/tags")
print(f"Status: {resp.status_code}")
tags = resp.json()
print(f"Total tags: {len(tags)}")
print("\nAll tags:")
for tag in tags:
    print(f"  id: {tag.get('id')} | label: {tag.get('label')} | slug: {tag.get('slug')}")

Status: 200
Total tags: 50

All tags:
  id: 101867 | label: product marekt fit | slug: product-marekt-fit
  id: 1512 | label: caitlin clark | slug: caitlin-clark
  id: 100601 | label: virgins | slug: virgins
  id: 101025 | label: Viktoria Plzen | slug: viktoria-plzen
  id: 101457 | label: Jerry After Dark | slug: jerry-after-dark
  id: 101655 | label: Wildfire | slug: wildfire
  id: 100344 | label: House Races | slug: house-races
  id: 100392 | label: redbull | slug: redbull
  id: 100257 | label: keith gill | slug: keith-gill
  id: 733 | label: haniyeh | slug: haniyeh
  id: 101896 | label: bat | slug: bat
  id: 100626 | label: Europa League | slug: europa-league
  id: 101026 | label: FDIC | slug: fdic
  id: 757 | label: legal cases | slug: legal-cases
  id: 933 | label: federal government | slug: federal-government
  id: 102136 | label: Timothée Chalamet | slug: timothee-chalamet
  id: 790 | label: controversies | slug: controversies
  id: 103813 | label: Major League Cricket | slug: m

In [10]:
# Test 10 — check what categories come back on events
resp = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"closed": "true", "limit": 50, "order": "volume", "ascending": "false"}
)
events = resp.json()
print(f"Total events returned: {len(events)}")

# Extract unique categories and tags
categories = set()
for e in events:
    cat = e.get("category")
    if cat:
        categories.add(cat)
    for tag in e.get("tags", []):
        print(f"  tag: {tag.get('id')} | {tag.get('label')} | {tag.get('slug')}")

print("\nUnique categories:")
for c in sorted(categories):
    print(f"  {c}")

# Also look at one event in full to understand structure
print("\nSample event keys:")
print(events[0].keys())
print("\nSample event:")
for k, v in events[0].items():
    if k != "markets":
        print(f"  {k}: {v}")

Total events returned: 50
  tag: 126 | Trump | trump
  tag: 2 | Politics | politics
  tag: 1101 | US Election | us-presidential-election
  tag: 15 | Joe Biden | joe-biden
  tag: 142 | nikki haley | nikki-haley
  tag: 1102 | 2024 us presidential election | 2024-us-presidential-election
  tag: 144 | Elections | elections
  tag: 177 | gavin newsom | gavin-newsom
  tag: 185 | 2024 election | 2024-election
  tag: 589 | us presidential election 2024 | us-presidential-election-2024
  tag: 360 | robert f. kennedy jr. | robert-fpt-kennedy-jrpt
  tag: 160 | ron desantis | ron-desantis
  tag: 220 | kamala harris | kamala-harris
  tag: 55 | vivek ramaswamy | vivek-ramaswamy
  tag: 945 | michelle obama | michelle-obama
  tag: 214 | hillary clinton | hillary-clinton
  tag: 382 | chris christie | chris-christie
  tag: 1103 | aoc | aoc
  tag: 704 | elizabeth warren | elizabeth-warren
  tag: 1104 | bernie sanders | bernie-sanders
  tag: 57 | 2024 presidential election | 2024-presidential-election
  tag

In [11]:
# Test 11 — find series_ticker in our Kalshi markets
import pandas as pd
kalshi_markets = pd.read_parquet("../data/raw/markets_kalshi.parquet")

# Check the raw market data for series_ticker field
print("Fields we have:", kalshi_markets.columns.tolist())
print("\nSample market_ids and categories:")
print(kalshi_markets[["market_id", "category"]].head(10))

# Also check the raw response for a field we might have missed
import requests
resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/markets",
    params={"status": "settled", "limit": 3}
)
data = resp.json()
sample = data["markets"][0]
print("\nAll fields in a raw Kalshi market:")
for k, v in sample.items():
    print(f"  {k}: {v}")

Fields we have: ['market_id', 'question', 'category', 'start_date', 'end_date', 'resolution_source', 'volume', 'liquidity', 'outcome', 'platform']

Sample market_ids and categories:
                     market_id              category
0  KXTEMPNYCH-26JUN0116-T77.99  KXTEMPNYCH-26JUN0116
1  KXTEMPNYCH-26JUN0116-T76.99  KXTEMPNYCH-26JUN0116
2  KXTEMPNYCH-26JUN0116-T75.99  KXTEMPNYCH-26JUN0116
3  KXTEMPNYCH-26JUN0116-T74.99  KXTEMPNYCH-26JUN0116
4  KXTEMPNYCH-26JUN0116-T73.99  KXTEMPNYCH-26JUN0116
5  KXTEMPNYCH-26JUN0116-T72.99  KXTEMPNYCH-26JUN0116
6  KXTEMPNYCH-26JUN0116-T71.99  KXTEMPNYCH-26JUN0116
7  KXTEMPNYCH-26JUN0116-T70.99  KXTEMPNYCH-26JUN0116
8  KXTEMPNYCH-26JUN0116-T69.99  KXTEMPNYCH-26JUN0116
9  KXTEMPNYCH-26JUN0116-T68.99  KXTEMPNYCH-26JUN0116

All fields in a raw Kalshi market:
  can_close_early: True
  close_time: 2026-06-01T22:17:18Z
  created_time: 2026-06-01T22:09:21.18759Z
  custom_strike: {'Associated Events': 'KXATPMATCH-26JUN01TIAARN,KXETH15M-26JUN011815,KXXRP15M-26

In [12]:
# Test 12 — test candlestick endpoint with a real non-parlay market
import time

# Use a market we know has trades
test_ticker = "KXMLBTOTAL-26MAY301605SDWSH-14"
series_ticker = "KXMLBTOTAL"  # everything before first hyphen+date

print(f"Testing: {test_ticker}")
print(f"Series: {series_ticker}")

import time
now = int(time.time())
one_week_ago = now - (7 * 24 * 3600)

resp = requests.get(
    f"https://external-api.kalshi.com/trade-api/v2/series/{series_ticker}/markets/{test_ticker}/candlesticks",
    params={
        "start_ts": one_week_ago,
        "end_ts": now,
        "period_interval": 60
    }
)
print(f"Status: {resp.status_code}")
print(resp.text[:500])

Testing: KXMLBTOTAL-26MAY301605SDWSH-14
Series: KXMLBTOTAL
Status: 200
{"candlesticks":[{"end_period_ts":1780182000,"open_interest_fp":"689.39","price":{"close_dollars":"0.0100","high_dollars":"0.7600","low_dollars":"0.0100","mean_dollars":"0.1105","open_dollars":"0.2100"},"volume_fp":"866.22","yes_ask":{"close_dollars":"1.0000","high_dollars":"1.0000","low_dollars":"0.0100","open_dollars":"1.0000"},"yes_bid":{"close_dollars":"0.0000","high_dollars":"0.2200","low_dollars":"0.0000","open_dollars":"0.1700"}}],"ticker":"KXMLBTOTAL-26MAY301605SDWSH-14"}


In [14]:
# Test — try startTs/endTs params, and verify our token is correct
test_token = "53135072462907880191400140706440867753044989936304433583131786753949599718775"

# Try with explicit wide date range
resp = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": test_token,
        "interval": "max",
        "fidelity": 1,
        "startTs": 1600000000,  # Sept 2020
        "endTs": 1800000000     # well into the future
    }
)
print(f"With startTs/endTs: {len(resp.json().get('history', []))} points")

# Also verify this token actually belongs to a real market
# by checking it against the Gamma API
resp2 = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={"clob_token_ids": test_token}
)
print(f"\nGamma API lookup status: {resp2.status_code}")
print(resp2.text[:300])

# Also try the NO token (index 1) from the same market
no_token = "60869871469376321574904667328762911501870754872924453995477779862968218702336"
resp3 = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={"market": no_token, "interval": "max", "fidelity": 1}
)
print(f"\nNO token history: {len(resp3.json().get('history', []))} points")

With startTs/endTs: 0 points

Gamma API lookup status: 200
[]


NO token history: 0 points


In [16]:
# Test — find a recently resolved market and check its price history
# Get recently closed markets sorted by close time
resp = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={"closed": "true", "order": "end_date", "ascending": "false", "limit": 5}
)
markets = resp.json()
for m in markets:
    print(f"Question: {m.get('question')[:60]}")
    print(f"End date: {m.get('endDate')}")
    tokens = m.get('clobTokenIds', '[]')
    import json
    if isinstance(tokens, str):
        tokens = json.loads(tokens)
    print(f"Tokens: {tokens[:1]}")
    
    if tokens:
        resp2 = requests.get(
            "https://clob.polymarket.com/prices-history",
            params={"market": tokens[0], "interval": "max", "fidelity": 60}
        )
        count = len(resp2.json().get("history", []))
        print(f"Price history points: {count}")
    print()

Question: Will Joe Biden get Coronavirus before the election?
End date: 2020-11-04T00:00:00Z
Tokens: ['53135072462907880191400140706440867753044989936304433583131786753949599718775']
Price history points: 0

Question: Will Airbnb begin publicly trading before Jan 1, 2021?
End date: 2021-01-02T00:00:00Z
Tokens: ['23957885615115430922384185661294483989521212430808224513177413172438775950057']
Price history points: 0

Question: Will a new Supreme Court Justice be confirmed before Nov 3rd
End date: 2020-11-04T00:00:00Z
Tokens: ['7192973530925903015287004813226256961086727269646918334118953338356315960259']
Price history points: 0

Question: Will Kim Kardashian and Kanye West divorce before Jan 1, 202
End date: 2021-01-02T00:00:00Z
Tokens: ['4485468797843270805361907688585033105060134197906764657766452761383944608702']
Price history points: 0

Question: Will Coinbase begin publicly trading before Jan 1, 2021?
End date: 2021-01-02T00:00:00Z
Tokens: ['36272374199536229966445059644094050673269

In [18]:
# Test — find markets that closed recently using closedTime
resp = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={"closed": "true", "order": "closed_time", "ascending": "false", "limit": 5}
)
markets = resp.json()
for m in markets:
    print(f"Question: {m.get('question', '')[:60]}")
    print(f"End date: {m.get('endDate')}")
    print(f"Closed time: {m.get('closedTime')}")
    tokens = m.get('clobTokenIds', '[]')
    if isinstance(tokens, str):
        import json
        tokens = json.loads(tokens)
    if tokens:
        resp2 = requests.get(
            "https://clob.polymarket.com/prices-history",
            params={"market": tokens[0], "interval": "max", "fidelity": 60}
        )
        count = len(resp2.json().get("history", []))
        print(f"Price history points: {count}")
    print()

Question: Will Joe Biden get Coronavirus before the election?
End date: 2020-11-04T00:00:00Z
Closed time: 2020-11-02 16:31:01+00
Price history points: 0

Question: Will Airbnb begin publicly trading before Jan 1, 2021?
End date: 2021-01-02T00:00:00Z
Closed time: 2020-12-11 20:53:24+00
Price history points: 0

Question: Will a new Supreme Court Justice be confirmed before Nov 3rd
End date: 2020-11-04T00:00:00Z
Closed time: 2020-10-27 00:40:55+00
Price history points: 0

Question: Will Kim Kardashian and Kanye West divorce before Jan 1, 202
End date: 2021-01-02T00:00:00Z
Closed time: 2021-01-02 21:35:34+00
Price history points: 0

Question: Will Coinbase begin publicly trading before Jan 1, 2021?
End date: 2021-01-02T00:00:00Z
Closed time: 2021-01-02 21:43:06+00
Price history points: 0



In [19]:
# Test — use events endpoint with closed=true and order by volume, check closedTime
resp = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"closed": "true", "order": "closed_time", "ascending": "false", "limit": 10}
)
events = resp.json()
for e in events:
    print(f"Title: {e.get('title', '')[:60]}")
    print(f"Closed time: {e.get('closedTime')}")
    markets = e.get("markets", [])
    if markets:
        tokens = markets[0].get("clobTokenIds", "[]")
        if isinstance(tokens, str):
            import json
            tokens = json.loads(tokens)
        if tokens:
            resp2 = requests.get(
                "https://clob.polymarket.com/prices-history",
                params={"market": tokens[0], "interval": "max", "fidelity": 60}
            )
            count = len(resp2.json().get("history", []))
            print(f"Price history points: {count}")
    print()

Title: NBA: Will the Mavericks beat the Grizzlies by more than 5.5 
Closed time: 2022-07-27T14:40:02.074Z
Price history points: 0

Title: NFL: Will the Falcons beat the Panthers by more than 3.5 poi
Closed time: 2022-07-27T14:40:02.139Z
Price history points: 0

Title: (In-Game Trading) Will the 49ers beat the Packers by more th
Closed time: 2022-07-27T14:40:02.191Z
Price history points: 0

Title: 2022 Norway Chess: Will Magnus Carlsen lose any game?
Closed time: 2022-07-27T14:40:02.236Z
Price history points: 0

Title: NBA: Will the Heat beat the Wizards by more than 7.5 points 
Closed time: 2022-07-27T14:40:02.284Z
Price history points: 0

Title: NFL: Will the Jaguars beat the Texans by more than 3.5 point
Closed time: 2022-07-27T14:40:02.343Z
Price history points: 0

Title: NBA: Will the Clippers beat the Celtics by more than 2.5 poi
Closed time: 2022-07-27T14:40:02.389Z
Price history points: 0

Title: NFL: Will the Chiefs beat the Bengals by more than 3.5 point
Closed time: 2022-07-2

In [20]:
# Test — try fetching events that were active recently but are now closed
# using a date filter if one exists
resp = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={
        "closed": "true",
        "order": "volume",
        "ascending": "false",
        "limit": 5,
        "end_date_min": "2024-01-01"  # try filtering by minimum end date
    }
)
events = resp.json()
for e in events:
    print(f"Title: {e.get('title','')[:60]}")
    print(f"Closed time: {e.get('closedTime')}")
    markets = e.get("markets", [])
    if markets:
        tokens = markets[0].get("clobTokenIds", "[]")
        if isinstance(tokens, str):
            import json
            tokens = json.loads(tokens)
        if tokens:
            resp2 = requests.get(
                "https://clob.polymarket.com/prices-history",
                params={"market": tokens[0], "interval": "max", "fidelity": 60}
            )
            count = len(resp2.json().get("history", []))
            print(f"Price history: {count} points")
    print()

Title: Presidential Election Winner 2024
Closed time: 2024-11-06T20:40:10Z
Price history: 0 points

Title: NBA Champion
Closed time: 2025-06-23T08:17:15Z
Price history: 0 points

Title: Super Bowl Champion 2025
Closed time: 2025-02-10T07:19:57Z
Price history: 0 points

Title: Champions League Winner
Closed time: 2025-06-01T00:05:52Z
Price history: 0 points

Title: Premier League Winner
Closed time: 2025-04-27T21:43:18Z
Price history: 0 points



In [21]:
resp = requests.get("https://docs.polymarket.com/llms.txt")
print(resp.text)

# Polymarket Documentation

## Docs

- [Negative Risk Markets](https://docs.polymarket.com/advanced/neg-risk.md): Capital-efficient trading for multi-outcome events
- [Authentication](https://docs.polymarket.com/api-reference/authentication.md): How to authenticate requests to the CLOB API
- [Create bridge addresses](https://docs.polymarket.com/api-reference/bridge/create-bridge-addresses.md)
- [Create withdrawal addresses](https://docs.polymarket.com/api-reference/bridge/create-withdrawal-addresses.md)
- [Get a quote](https://docs.polymarket.com/api-reference/bridge/get-a-quote.md)
- [Get supported assets](https://docs.polymarket.com/api-reference/bridge/get-supported-assets.md)
- [Get transaction status](https://docs.polymarket.com/api-reference/bridge/get-transaction-status.md)
- [Get aggregated builder leaderboard](https://docs.polymarket.com/api-reference/builders/get-aggregated-builder-leaderboard.md)
- [Get daily builder volume time-series](https://docs.polymarket.com/api-refere

In [22]:
resp = requests.get("https://docs.polymarket.com/api-reference/core/get-trades-for-a-user-or-markets.md")
print(resp.text)

> ## Documentation Index
> Fetch the complete documentation index at: https://docs.polymarket.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Get trades for a user or markets



## OpenAPI

````yaml /api-spec/data-openapi.yaml get /trades
openapi: 3.0.3
info:
  title: Polymarket Data API
  version: 1.0.0
  description: >
    HTTP API for Polymarket data. This specification documents all public
    routes.
servers:
  - url: https://data-api.polymarket.com
    description: Relative server (same host)
security: []
tags:
  - name: Data API Status
    description: Data API health check
  - name: Core
  - name: Builders
  - name: Misc
paths:
  /trades:
    get:
      tags:
        - Core
      summary: Get trades for a user or markets
      parameters:
        - in: query
          name: limit
          schema:
            type: integer
            default: 100
            minimum: 0
            maximum: 10000
        - in: query
          name: offs

In [23]:
# Test — use market parameter exactly as documented (comma-separated condition IDs)
test_condition_id = "0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa00481764d320e7415f7a9"  # Biden COVID market

resp = requests.get(
    "https://data-api.polymarket.com/trades",
    params={
        "market": test_condition_id,
        "limit": 10,
        "offset": 0,
        "takerOnly": "false"
    }
)
print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Trades returned: {len(data)}")
if data:
    print(f"First trade conditionId: {data[0].get('conditionId')}")
    print(f"First trade timestamp: {data[0].get('timestamp')}")
    print(f"Title: {data[0].get('title')}")

Status: 200
Trades returned: 0


In [24]:
# Test — try eventId filter instead of market/conditionId
# We need the numeric event ID for the Biden COVID market
# From earlier we saw id: "12" in the Gamma response

resp = requests.get(
    "https://data-api.polymarket.com/trades",
    params={
        "eventId": 12,
        "limit": 10,
        "takerOnly": "false"
    }
)
print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Trades returned: {len(data)}")
if data:
    print(f"conditionId: {data[0].get('conditionId')}")
    print(f"timestamp: {data[0].get('timestamp')}")
    print(f"title: {data[0].get('title')}")
else:
    print(resp.text[:200])

Status: 200
Trades returned: 0
[]


In [26]:
from datetime import datetime

# 2024 Presidential Election - Kamala Harris popular vote token
token_id = "21271000291843361249209065706097167029083067325856089903026951915683588703117"

start_time = int(datetime(2024, 11, 1).timestamp())
end_time = int(datetime(2024, 11, 14).timestamp())

resp = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": token_id,
        "startTs": start_time,
        "endTs": end_time,
        "fidelity": 60
    }
)
data = resp.json()
history = data.get("history", [])
print(f"Data points: {len(history)}")
if history:
    df = pd.DataFrame(history)
    df["t"] = pd.to_datetime(df["t"], unit="s", utc=True)
    print(f"Date range: {df['t'].min()} to {df['t'].max()}")
    print(df.head())

Data points: 267
Date range: 2024-11-01 07:00:02+00:00 to 2024-11-12 10:00:03+00:00
                          t       p
0 2024-11-01 07:00:02+00:00  0.6365
1 2024-11-01 08:00:02+00:00  0.6365
2 2024-11-01 09:00:03+00:00  0.6345
3 2024-11-01 10:00:02+00:00  0.6195
4 2024-11-01 11:00:02+00:00  0.6185


In [27]:
from datetime import datetime

# Trump win token from Presidential Election 2024
# Get it from the Gamma API first
resp = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"slug": "presidential-election-winner-2024"}
)
event = resp.json()[0]
print(f"Event: {event['title']}")
print(f"Start: {event['startDate']}")
print(f"End: {event['endDate']}")
print(f"Closed: {event['closedTime']}")
print()
for m in event["markets"]:
    import json
    tokens = m.get("clobTokenIds", "[]")
    if isinstance(tokens, str):
        tokens = json.loads(tokens)
    print(f"Question: {m.get('question')}")
    print(f"Tokens: {tokens}")
    print()

Event: Presidential Election Winner 2024
Start: 2024-01-04T22:58:00Z
End: 2024-11-05T12:00:00Z
Closed: 2024-11-06T20:40:10Z

Question: Will Donald Trump win the 2024 US Presidential Election?
Tokens: ['21742633143463906290569050155826241533067272736897614950488156847949938836455', '48331043336612883890938759509493159234755048973500640148014422747788308965732']

Question: Will Joe Biden win the 2024 US Presidential Election?
Tokens: ['88027839609243624193415614179328679602612916497045596227438675518749602824929', '34731657770883441140875001518098751138877095477683682718012432921110142479972']

Question: Will Nikki Haley win the 2024 US Presidential Election?
Tokens: ['19083349462791593334532840548890602187185739923311385087650426802477691161360', '25663677275476030658483179785762851061160843737234225579491314980654272946621']

Question: Will Gavin Newsom win the 2024 US Presidential Election?
Tokens: ['99200347365169760700385453164878188504479548439905371494493482364634358863823', '8806

In [28]:
from datetime import datetime, timedelta

trump_token = "21742633143463906290569050155826241533067272736897614950488156847949938836455"

# Fetch in 15-day chunks from Jan 2024 to Nov 2024
start_date = datetime(2024, 1, 4)
end_date = datetime(2024, 11, 6)

all_history = []
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + timedelta(days=15), end_date)
    
    resp = requests.get(
        "https://clob.polymarket.com/prices-history",
        params={
            "market": trump_token,
            "startTs": int(current_start.timestamp()),
            "endTs": int(current_end.timestamp()),
            "fidelity": 60
        }
    )
    history = resp.json().get("history", [])
    all_history.extend(history)
    current_start = current_end
    time.sleep(0.2)

df = pd.DataFrame(all_history)
df["t"] = pd.to_datetime(df["t"], unit="s", utc=True)
print(f"Total data points: {len(df)}")
print(f"Date range: {df['t'].min()} to {df['t'].max()}")
print(f"Price range: {df['p'].min():.3f} to {df['p'].max():.3f}")
print(df.head())

Total data points: 7348
Date range: 2024-01-05 00:00:03+00:00 to 2024-11-06 06:00:02+00:00
Price range: 0.395 to 0.987
                          t      p
0 2024-01-05 00:00:03+00:00  0.500
1 2024-01-05 01:00:03+00:00  0.500
2 2024-01-05 02:00:03+00:00  0.420
3 2024-01-05 03:00:03+00:00  0.410
4 2024-01-05 04:00:03+00:00  0.405


In [30]:
# Test — historical candlesticks endpoint (no series_ticker needed)
test_ticker = "KXMLBTOTAL-26MAY301605SDWSH-14"

import time as time_module
now = int(time_module.time())
thirty_days_ago = now - (30 * 24 * 3600)

resp = requests.get(
    f"https://external-api.kalshi.com/trade-api/v2/historical/markets/{test_ticker}/candlesticks",
    params={
        "start_ts": thirty_days_ago,
        "end_ts": now,
        "period_interval": 60
    }
)
print(f"Status: {resp.status_code}")
print(resp.text[:300])

Status: 404
{"error":{"code":"failed_to_get_market_by_ticker:_not_found","message":"failed to get market by ticker: not found","service":"query-exchange"}}


In [35]:
# Get markets closed before June 1 2026, filter to after April 2 in Python
june_1 = 1748736000
april_2 = 1743552000

resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/markets",
    params={
        "status": "settled",
        "limit": 100,
        "max_close_ts": june_1
    }
)
data = resp.json()
markets = data.get("markets", [])
print(f"Total markets returned: {len(markets)}")

# Filter to after April 2 in Python
import pandas as pd
df = pd.DataFrame(markets)
df["close_time"] = pd.to_datetime(df["close_time"], utc=True)
cutoff = pd.Timestamp("2026-04-02", tz="UTC")
filtered = df[df["close_time"] > cutoff]
print(f"Markets settled after April 2: {len(filtered)}")
print(filtered[["ticker", "close_time", "volume_fp"]].head(10))

Total markets returned: 68
Markets settled after April 2: 0
Empty DataFrame
Columns: [ticker, close_time, volume_fp]
Index: []


In [39]:
# Search for known high-value Kalshi series directly
series_to_try = ["KXFED", "KXINFL", "KXGDP", "KXUNEMP", "KXPRES", "KXSENATE", "KXHOUSE"]

for series in series_to_try:
    resp = requests.get(
        "https://external-api.kalshi.com/trade-api/v2/markets",
        params={"status": "settled", "limit": 3, "series_ticker": series}
    )
    data = resp.json()
    markets = data.get("markets", [])
    if markets:
        for m in markets:
            print(f"{m.get('ticker')} | volume: {m.get('volume_fp')} | close: {m.get('close_time')}")
    else:
        print(f"{series}: no markets found")
    print()

KXFED-26APR-T5.25 | volume: 29256.24 | close: 2026-04-29T17:55:00Z
KXFED-26APR-T5.00 | volume: 20249.20 | close: 2026-04-29T17:55:00Z
KXFED-26APR-T4.75 | volume: 19372.21 | close: 2026-04-29T17:55:00Z

KXINFL: no markets found

KXGDP-26APR30-T4.5 | volume: 49452.49 | close: 2026-04-30T12:29:00Z
KXGDP-26APR30-T4.0 | volume: 69310.99 | close: 2026-04-30T12:29:00Z
KXGDP-26APR30-T3.5 | volume: 78413.43 | close: 2026-04-30T12:29:00Z

KXUNEMP: no markets found

KXPRES: no markets found

KXSENATE: no markets found

KXHOUSE: no markets found



In [40]:
# Get all KXFED and KXGDP markets and test candlesticks
for series in ["KXFED", "KXGDP"]:
    resp = requests.get(
        "https://external-api.kalshi.com/trade-api/v2/markets",
        params={"status": "settled", "limit": 100, "series_ticker": series}
    )
    markets = resp.json().get("markets", [])
    print(f"{series}: {len(markets)} markets")
    
    # Test candlesticks on the first one
    if markets:
        m = markets[0]
        ticker = m.get("ticker")
        series_ticker = ticker.split("-")[0]
        
        now = int(time_module.time())
        one_year_ago = now - (365 * 24 * 3600)
        
        resp2 = requests.get(
            f"https://external-api.kalshi.com/trade-api/v2/series/{series_ticker}/markets/{ticker}/candlesticks",
            params={"start_ts": one_year_ago, "end_ts": now, "period_interval": 60}
        )
        candles = resp2.json().get("candlesticks", [])
        print(f"  Test ticker: {ticker}")
        print(f"  Candlesticks: {len(candles)}")
        if candles:
            print(f"  First candle: {candles[0]}")
    print()

KXFED: 11 markets
  Test ticker: KXFED-26APR-T5.25
  Candlesticks: 0

KXGDP: 8 markets
  Test ticker: KXGDP-26APR30-T4.5
  Candlesticks: 0



In [43]:
# First recheck the actual cutoff
resp = requests.get("https://external-api.kalshi.com/trade-api/v2/historical/cutoff")
print("Cutoff:", resp.json())

# Then try regular candlesticks endpoint with narrow range around close date
test_ticker = "KXFED-26APR-T5.25"
close_ts = 1745946900  # April 29 2026
start_ts = close_ts - (30 * 24 * 3600)  # 30 days before close

resp2 = requests.get(
    f"https://external-api.kalshi.com/trade-api/v2/series/KXFED/markets/{test_ticker}/candlesticks",
    params={"start_ts": start_ts, "end_ts": close_ts, "period_interval": 60}
)
print(f"\nRegular endpoint status: {resp2.status_code}")
candles = resp2.json().get("candlesticks", [])
print(f"Candlesticks: {len(candles)}")
if candles:
    print(f"First: {candles[0]}")

Cutoff: {'market_settled_ts': '2026-04-02T00:00:00Z', 'orders_updated_ts': '2026-04-02T00:00:00Z', 'trades_created_ts': '2026-04-02T00:00:00Z'}

Regular endpoint status: 200
Candlesticks: 0


In [44]:
resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/markets/trades",
    params={"ticker": "KXFED-26APR-T5.25", "limit": 10}
)
print(f"Status: {resp.status_code}")
data = resp.json()
trades = data.get("trades", [])
print(f"Trades: {len(trades)}")
if trades:
    print(trades[0])

Status: 200
Trades: 10
{'count_fp': '50.00', 'created_time': '2026-04-29T08:13:27.176107Z', 'no_price_dollars': '0.9900', 'taker_book_side': 'bid', 'taker_outcome_side': 'yes', 'taker_side': 'yes', 'ticker': 'KXFED-26APR-T5.25', 'trade_id': '4c6f1ad8-ce58-4f70-0378-574f88468cf8', 'yes_price_dollars': '0.0100'}


In [46]:
# Get all KXFED markets and find ones with price movement
resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/markets",
    params={"status": "settled", "limit": 100, "series_ticker": "KXFED"}
)
markets = resp.json().get("markets", [])

for m in markets:
    ticker = m.get("ticker")
    # Skip if very low volume
    vol = float(m.get("volume_fp") or 0)
    if vol < 5000:
        continue
        
    resp2 = requests.get(
        "https://external-api.kalshi.com/trade-api/v2/markets/trades",
        params={"ticker": ticker, "limit": 1000}
    )
    trades = resp2.json().get("trades", [])
    if not trades:
        continue
    
    prices = [float(t["yes_price_dollars"]) for t in trades]
    price_range = max(prices) - min(prices)
    
    print(f"{ticker} | vol: {vol:,.0f} | price range: {min(prices):.3f}-{max(prices):.3f} | range: {price_range:.3f} | trades: {len(trades)}")

KXFED-26APR-T5.25 | vol: 29,256 | price range: 0.010-0.010 | range: 0.000 | trades: 92
KXFED-26APR-T5.00 | vol: 20,249 | price range: 0.010-0.010 | range: 0.000 | trades: 86
KXFED-26APR-T4.75 | vol: 19,372 | price range: 0.010-0.010 | range: 0.000 | trades: 72
KXFED-26APR-T4.50 | vol: 19,436 | price range: 0.010-0.010 | range: 0.000 | trades: 102
KXFED-26APR-T4.25 | vol: 213,360 | price range: 0.010-0.030 | range: 0.020 | trades: 440
KXFED-26APR-T4.00 | vol: 923,511 | price range: 0.010-0.010 | range: 0.000 | trades: 1000
KXFED-26APR-T3.75 | vol: 3,055,287 | price range: 0.010-0.020 | range: 0.010 | trades: 1000
KXFED-26APR-T3.50 | vol: 124,409 | price range: 0.960-0.990 | range: 0.030 | trades: 581
KXFED-26APR-T3.25 | vol: 227,314 | price range: 0.980-0.990 | range: 0.010 | trades: 165
KXFED-26APR-T3.00 | vol: 369,024 | price range: 0.980-0.990 | range: 0.010 | trades: 102
KXFED-26APR-T2.75 | vol: 55,890 | price range: 0.990-0.990 | range: 0.000 | trades: 88


In [47]:
# Check if there are KXFED markets from earlier meetings
resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/markets",
    params={"status": "settled", "limit": 100, "series_ticker": "KXFED"}
)
markets = resp.json().get("markets", [])
for m in markets:
    print(f"{m.get('ticker')} | close: {m.get('close_time')} | volume: {m.get('volume_fp')}")

KXFED-26APR-T5.25 | close: 2026-04-29T17:55:00Z | volume: 29256.24
KXFED-26APR-T5.00 | close: 2026-04-29T17:55:00Z | volume: 20249.20
KXFED-26APR-T4.75 | close: 2026-04-29T17:55:00Z | volume: 19372.21
KXFED-26APR-T4.50 | close: 2026-04-29T17:55:00Z | volume: 19436.36
KXFED-26APR-T4.25 | close: 2026-04-29T17:55:00Z | volume: 213360.22
KXFED-26APR-T4.00 | close: 2026-04-29T17:55:00Z | volume: 923510.63
KXFED-26APR-T3.75 | close: 2026-04-29T17:55:00Z | volume: 3055286.80
KXFED-26APR-T3.50 | close: 2026-04-29T17:55:00Z | volume: 124408.99
KXFED-26APR-T3.25 | close: 2026-04-29T17:55:00Z | volume: 227314.31
KXFED-26APR-T3.00 | close: 2026-04-29T17:55:00Z | volume: 369024.19
KXFED-26APR-T2.75 | close: 2026-04-29T17:55:00Z | volume: 55890.32


In [48]:
# Step 1 — check historical markets endpoint for KXFED
resp = requests.get(
    "https://external-api.kalshi.com/trade-api/v2/historical/markets",
    params={"limit": 100, "series_ticker": "KXFED"}
)
print(f"Status: {resp.status_code}")
data = resp.json()
markets = data.get("markets", [])
print(f"Markets found: {len(markets)}")
for m in markets:
    print(f"  {m.get('ticker')} | settled: {m.get('settlement_ts')} | volume: {m.get('volume_fp')}")

Status: 200
Markets found: 100
  KXFED-26MAR-T5.25 | settled: 2026-03-18T18:13:09.22773Z | volume: 4641.00
  KXFED-26MAR-T5.00 | settled: 2026-03-18T18:13:09.22773Z | volume: 1880.00
  KXFED-26MAR-T4.75 | settled: 2026-03-18T18:13:09.22773Z | volume: 6652.00
  KXFED-26MAR-T4.50 | settled: 2026-03-18T18:13:09.22773Z | volume: 10659.00
  KXFED-26MAR-T4.25 | settled: 2026-03-18T18:13:09.22773Z | volume: 22014.00
  KXFED-26MAR-T4.00 | settled: 2026-03-18T18:13:09.22773Z | volume: 149225.00
  KXFED-26MAR-T3.75 | settled: 2026-03-18T18:13:09.22773Z | volume: 930486.00
  KXFED-26MAR-T3.50 | settled: 2026-03-18T18:13:09.22773Z | volume: 125802.00
  KXFED-26MAR-T3.25 | settled: 2026-03-18T18:13:09.22773Z | volume: 119466.00
  KXFED-26MAR-T3.00 | settled: 2026-03-18T18:13:09.22773Z | volume: 114101.00
  KXFED-26MAR-T2.75 | settled: 2026-03-18T18:13:09.22773Z | volume: 218524.00
  KXFED-26JAN-T5.25 | settled: 2026-01-28T19:45:49.986017Z | volume: 13274.00
  KXFED-26JAN-T5.00 | settled: 2026-01-28

In [49]:
test_ticker = "FED-25MAY-T4.25"  # 3.7M volume, high stakes

resp = requests.get(
    f"https://external-api.kalshi.com/trade-api/v2/historical/markets/{test_ticker}/candlesticks",
    params={
        "start_ts": 1744500000,  # roughly April 13 2025
        "end_ts": 1746650000,    # roughly May 7 2025
        "period_interval": 60
    }
)
print(f"Status: {resp.status_code}")
data = resp.json()
candles = data.get("candlesticks", [])
print(f"Candlesticks: {len(candles)}")
if candles:
    print(f"First: {candles[0]}")
    print(f"Last: {candles[-1]}")

Status: 200
Candlesticks: 477
First: {'end_period_ts': 1744502400, 'open_interest': '655723.00', 'price': {'close': '0.7900', 'high': '0.7900', 'low': '0.7700', 'mean': '0.7797', 'open': '0.7800', 'previous': '0.7700'}, 'volume': '102.00', 'yes_ask': {'close': '0.7900', 'high': '0.7900', 'low': '0.7800', 'open': '0.7800'}, 'yes_bid': {'close': '0.7800', 'high': '0.7800', 'low': '0.7700', 'open': '0.7700'}}
Last: {'end_period_ts': 1746640800, 'open_interest': '1107911.00', 'price': {'close': '0.9700', 'high': '0.9800', 'low': '0.9700', 'mean': '0.9759', 'open': '0.9700', 'previous': '0.9800'}, 'volume': '4077.00', 'yes_ask': {'close': '1.0000', 'high': '1.0000', 'low': '0.9800', 'open': '0.9800'}, 'yes_bid': {'close': '0.0000', 'high': '0.9700', 'low': '0.0000', 'open': '0.9700'}}
